**LangGraph**

In [ ]:
import os, sys
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root) 
from agents.flight.agent import build_flight_graph 

# Khởi tạo graph (cần dùng await vì hàm này là async)
graph = await build_flight_graph()

**Test**

In [ ]:
from IPython.display import Image, display

try:
    display(Image(graph.get_graph(xray=True).draw_mermaid_png()))
except Exception:
    pass
# 1. In ra mã nguồn Mermaid của graph
mermaid_code = graph.get_graph(xray=True).draw_mermaid()
print(mermaid_code)

In [ ]:
import json
from langchain_core.messages import AIMessage


def _strip_noise(obj):
    """Bỏ detailToken / field dài để dễ đọc khi in notebook."""
    if isinstance(obj, dict):
        return {
            k: _strip_noise(v)
            for k, v in obj.items()
            if k not in ("detailToken", "signature", "extras")
        }
    if isinstance(obj, list):
        return [_strip_noise(x) for x in obj]
    return obj


def format_content(content, max_len: int = 2500) -> str:
    """Gemini đôi khi trả content dạng list[{type,text,extras}] — lấy text sạch."""
    if content is None:
        return ""
    if isinstance(content, str):
        text = content
    elif isinstance(content, list):
        parts = []
        for block in content:
            if isinstance(block, dict):
                if "text" in block:
                    parts.append(str(block["text"]))
                else:
                    parts.append(json.dumps(_strip_noise(block), ensure_ascii=False)[:200])
            else:
                parts.append(str(block))
        text = "\n".join(parts)
    elif isinstance(content, dict):
        text = content.get("text") or json.dumps(_strip_noise(content), ensure_ascii=False)
    else:
        text = str(content)

    text = text.strip()
    # Nếu là JSON (tool output), pretty-print + bỏ token dài
    if text.startswith("{") or text.startswith("["):
        try:
            parsed = json.loads(text)
            text = json.dumps(_strip_noise(parsed), ensure_ascii=False, indent=2)
        except Exception:
            pass

    if len(text) > max_len:
        return text[:max_len] + "\n... (truncated)"
    return text


config = {"configurable": {"thread_id": "user_12345"}}


async def stream_graph_updates(inputs: dict):
    async for event in graph.astream(inputs, config=config):
        if "flight_chat" in event:
            for msg in event["flight_chat"].get("messages", []):
                if isinstance(msg, AIMessage) and msg.tool_calls:
                    for tc in msg.tool_calls:
                        print("🔧 Tool:", tc["name"])
                        print("📥 Args:", tc["args"])
                text = format_content(getattr(msg, "content", None))
                if text:
                    print("\n🤖 Assistant:\n" + text)

        if "tools" in event:
            for msg in event["tools"].get("messages", []):
                text = format_content(getattr(msg, "content", None), max_len=1500)
                name = getattr(msg, "name", None) or "tool"
                if text:
                    print(f"\n📤 Tool Output ({name}):\n{text}")
        print("-" * 50)


In [ ]:
test_cases = [

    "Tôi muốn kiếm chuyen bay từ tp hcm đến Tuy Hòa, bay thẳng, ngày 21/10/2026."
]


for q in test_cases:
    print(f"\n🧑‍💻 User: {q}")
    inputs = {"messages": [{"role": "user", "content": q}]}
    print(inputs)
    await stream_graph_updates(inputs) 

In [ ]:
test_cases = [

    # 4) Roundtrip
    "Tìm vé khứ hồi từ Sài Gòn đi Phú Quốc, đi 10/11/2026 về 15/11/2026, bay thẳng nếu có.",


]

for q in test_cases:
    print("=" * 60)
    print(f"🧑‍💻 User: {q}")
    inputs = {"messages": [{"role": "user", "content": q}]}
    await stream_graph_updates(inputs)

In [ ]:
test_cases = [
   "Tìm vé khứ hồi SGN đi Tuy Hòa ngày 21/8/2026, về ngày 25/8/2026."
]

# --- Đoạn code chạy test (giữ nguyên) ---
for q in test_cases:
    print(f"==================================================")
    print(f"🧑‍💻 User: {q}")
    inputs = {"messages": [{"role": "user", "content": q}]}
    # Giả sử bạn có hàm stream_graph_updates để chạy graph
    await stream_graph_updates(inputs)
    print(f"==================================================\n")


In [ ]:
test_cases = [
   "Tìm vé khứ hồi SGN đi Tuy Hòa ngày 21/7/2026, về ngày 25/7/2026, khoi hanh chieu di khoang 7 gio sang."
]

# --- Đoạn code chạy test (giữ nguyên) ---
for q in test_cases:
    print(f"==================================================")
    print(f"🧑‍💻 User: {q}")
    inputs = {"messages": [{"role": "user", "content": q}]}
    # Giả sử bạn có hàm stream_graph_updates để chạy graph
    await stream_graph_updates(inputs)
    print(f"==================================================\n")


In [ ]:
test_cases = [
   "Tìm vé khứ hồi SGN đi TBB ngày 21/7/2026, về ngày 25/7/2026, khoi hanh chieu di khoang 7 gio sang, chiều về khoảng 9 giờ sáng."
]

# --- Đoạn code chạy test (giữ nguyên) ---
for q in test_cases:
    print(f"==================================================")
    print(f"🧑‍💻 User: {q}")
    inputs = {"messages": [{"role": "user", "content": q}]}
    # Giả sử bạn có hàm stream_graph_updates để chạy graph
    await stream_graph_updates(inputs)
    print(f"==================================================\n")


In [ ]:
test_cases = [
   "Tôi muốn book FL-D5678A ."
]

# --- Đoạn code chạy test (giữ nguyên) ---
for q in test_cases:
    print(f"==================================================")
    print(f"🧑‍💻 User: {q}")
    inputs = {"messages": [{"role": "user", "content": q}]}
    # Giả sử bạn có hàm stream_graph_updates để chạy graph
    await stream_graph_updates(inputs)
    print(f"==================================================\n")
